# Exploring `ffe` — flat files into tables

This notebook walks the whole path: **a messy file → a table you can query**.

Run it top to bottom. It makes its own sample data in `_notebook_demo/`, so it
touches nothing of yours, and the last cell cleans up.

What you'll see:

1. What the sample files look like (they're deliberately awkward)
2. Asking the tool to *measure* a file — no AI involved
3. Reading a sample **without saving anything**
4. Running it for real, into a table you choose
5. Reading the table back
6. Tracing a row to the exact line it came from
7. Bad rows: where they go and why the job can refuse to save

In [ ]:
import shutil, sys, zipfile
from pathlib import Path

import polars as pl

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
pl.Config(fmt_str_lengths=40, tbl_rows=12)

WORK = ROOT / "_notebook_demo"
shutil.rmtree(WORK, ignore_errors=True)
(WORK / "drops").mkdir(parents=True)

print("project :", ROOT)
print("scratch :", WORK)

---
## 1. The files we're dealing with

Two shapes, both common in reference data, neither openable by a normal CSV reader.

**Shape A — banner-delimited.** Lines of `*` split the file into sections, and the
header describes its own columns in a mini-language (`name string(100)`).

**Shape B — tag per record.** Every row starts with a letter saying what kind of
row it is: `F` file header, `H` column names, `I` the actual data, `E` end.

In [ ]:
SHAPE_A = '''*
*
*
Header

name string(100)
age  int(10)
*
ABhishek,10
ANkita,20,
Nilanjana,30
*
END
'''

SHAPE_B = '''F|issue.{n}|20260822
H|20260822|issue|id|name
I|1|Abhishek
I|2|Nilanjana
I|3|Ankita
E|issue
'''

print("SHAPE A -- banner delimited")
print(SHAPE_A)
print("SHAPE B -- tag per record")
print(SHAPE_B.format(n=0))

Things that make these annoying, and that we'll watch the tool handle:

- In A, `*` is **both** the section separator *and* three lines of noise at the top.
- In A, `ANkita,20,` has a **trailing comma** — one field too many.
- In A the column types come from the file itself (`int(10)` → a number).
- In B, the date lives in the `F` row but every data row needs it.
- In B, the `I` at the start of each row is **not** data and must be dropped.

---
## 2. Measure the file before describing it

`profile` counts things. It does **not** call a model. Give it a small sample —
100 lines is plenty.

In [ ]:
from ffe.core.profile import profile

sample_a = WORK / "sample_a.txt"
sample_b = WORK / "sample_b.txt"
sample_a.write_text(SHAPE_A)
sample_b.write_text(SHAPE_B.format(n=0))

for path in (sample_a, sample_b):
    p = profile(path.read_bytes())
    print(f"{path.name}")
    print(f"   hint       : {p['structure_hint']['strategy']}")
    print(f"   because    : {p['structure_hint']['why']}")
    print(f"   delimiters : {p['delimiters']}")
    print(f"   row codes  : {p['first_field_tokens'] or '(none)'}")
    print()

It worked out both shapes from arithmetic alone:

- Shape A → `sentinel`, because 5 lines are nothing but `*`
- Shape B → `record_tag`, because field 0 holds a few short repeated codes **and**
  the row width changes depending on the code

That last part is what separates a tag file from a plain CSV — in a CSV every row
has the same number of columns.

---
## 3. Describe the file

A **feed** is one YAML file answering three questions: where files come from, what
they look like, and which table they go in.

Two already exist in `feeds/`. Let's read the one for shape B and see how it maps
onto what `profile` just told us.

In [ ]:
print((ROOT / "feeds" / "pipe-tagged-feed.yaml").read_text())

Line by line, the parts that matter:

| Setting | Meaning |
|---|---|
| `strategy: record_tag` | field 0 says what kind of row this is |
| `records: I: {kind: data}` | rows starting with `I` are the real data |
| `skip_leading_fields: 3` | in `H\|20260822\|issue\|id\|name`, skip 3 → columns are `id`, `name` |
| `drop_columns: [tag]` | throw away the `I` |
| `schema_override: {id: int64}` | make `id` a number, not text |
| `promote_fields: [business_date, file_id]` | copy those from the `F` row onto every data row |

---
## 4. Try it without saving anything

`dry-run` is safe by design — the parsing code has no ability to write files at
all. Run it as many times as you like while you get the description right.

In [ ]:
from ffe.core.engine import parse
from ffe.core.spec import FeedSpec

feed_b = FeedSpec.from_yaml(ROOT / "feeds" / "pipe-tagged-feed.yaml")
result = parse(feed_b.parser, sample_b.read_bytes())

print("THE TABLE:")
print(result.frame)
print("\nHOW IT WENT:")
r = result.report
print(f"   lines in file    : {r.lines_total}")
print(f"   rows of data     : {r.rows_parsed}")
print(f"   rows rejected    : {r.rows_rejected}")
print(f"   ragged lines     : {r.ragged_lines}")
print(f"   unknown row codes: {r.unknown_tags}")

Note what happened without being asked:

- `id` is a **number**, `name` is text
- the `I` tag is gone
- `business_date` and `file_id` were lifted off the `F` row onto every row
- `_src_line_no` records which line each row came from

Now shape A, the awkward one:

In [ ]:
feed_a = FeedSpec.from_yaml(ROOT / "feeds" / "banner-ddl-feed.yaml")
result_a = parse(feed_a.parser, sample_a.read_bytes())

print(result_a.frame)
print(f"\nage column type: {result_a.frame.schema['age']}   <- came from 'int(10)' in the file")
print(f"ragged lines   : {result_a.report.ragged_lines}   <- the trailing comma on ANkita, handled")
print(f"trailer found  : {result_a.report.trailer_seen}")

The three `*` lines at the top collapsed into one boundary rather than creating
three empty sections, and `ANkita,20,` was read correctly despite the extra comma
— but it was still **reported** as ragged, so you know it's there.

---
## 5. When the description is wrong

You don't get a stack trace. You get told which setting to change and what to
change it to. This is the part that makes the tool drivable by a coding assistant.

In [ ]:
from ffe.core.report import ParseError

broken = FeedSpec.from_yaml(ROOT / "feeds" / "banner-ddl-feed.yaml")
broken.parser.data.delimiter = "|"          # wrong on purpose

try:
    parse(broken.parser, sample_a.read_bytes())
except ParseError as e:
    for k, v in e.to_dict().items():
        print(f"{k:12}: {v}")

Read it as: *"the setting `parser.data.delimiter` is wrong; `,` gives 2 columns."*

`blame` decides what to do next:

- `spec` → **your description is wrong.** Fix it and retry.
- `file` → **the file is broken** (truncated, corrupt). Stop retrying, go ask the sender.

---
## 6. Run it for real — into a table you choose

Now let's make a zip with 12 files in it, and load it. **You pick the output
table** — either in the YAML (`target.table`) or by overriding it here.

In [ ]:
from ffe.io.runner import run

drop = WORK / "drops" / "issues_20260822.zip"
with zipfile.ZipFile(drop, "w", zipfile.ZIP_DEFLATED) as z:
    for n in range(12):
        z.writestr(f"issue_{n:03d}.txt", SHAPE_B.format(n=n))

feed = FeedSpec.from_yaml(ROOT / "feeds" / "pipe-tagged-feed.yaml")
feed.source.pattern = str(drop)
feed.target.table = "sandbox.issues"      # <-- your output table

job = run(feed, WORK / "ws", workers=4)

print(f"status        : {job.status}")
print(f"files read    : {job.members}   (in parallel)")
print(f"rows saved    : {job.rows}")
print(f"rows rejected : {job.rejects}")
print(f"saved into    : {job.commit['table']}")
print(f"parquet files : {job.commit['files']}")
print(f"snapshots     : 1  (id {job.commit['snapshot_id']})")

12 files were read at the same time by 4 workers, each writing its own file — and
then **one single save** registered all 12 at once. That last part is deliberate:
letting 12 workers write to the table simultaneously is what causes conflicts and
retries, so they never touch it.

---
## 7. Read the table back

This is a real Iceberg table. Here it's read into Polars, but anything that speaks
Iceberg can read it.

In [ ]:
from ffe.io.sink import scan

df = scan(WORK / "ws" / "warehouse", "sandbox.issues")

print(f"{len(df)} rows, {len(df.columns)} columns\n")
print(df.select(["id", "name", "business_date", "file_id"]).head(6))

In [ ]:
print("your columns:")
print("   ", [c for c in df.columns if not c.startswith("_")])
print("\nadded for you:")
for c in [c for c in df.columns if c.startswith("_")]:
    print(f"    {c:16} {df.schema[c]}")

---
## 8. Where did this row come from?

Every row remembers its file **and its line number**. When someone asks about a
strange value six months from now, this is the answer.

In [ ]:
row = df.row(5, named=True)
print("a row from the table:")
print(f"   id={row['id']}  name={row['name']}")
print(f"   came from : {row['_src_file']}")
print(f"   line      : {row['_src_line_no']}")

# go and look at that exact line in the original zip
archive, _, member = row["_src_file"].partition("!")
with zipfile.ZipFile(WORK / "drops" / archive) as z:
    original = z.read(member).decode().splitlines()

print(f"\nline {row['_src_line_no']} of {member} really is:")
print(f"   {original[row['_src_line_no'] - 1]!r}")

In [ ]:
# which file did each row come from? one row per input file
print(df.group_by("_src_file").len().sort("_src_file").head(6))

---
## 9. Bad rows

Now a deliberately broken drop: one good file and three where `id` isn't a number.

Two things to watch: bad rows go somewhere useful, **and** if too many rows are
bad the job refuses to save anything at all.

In [ ]:
BROKEN = '''F|issue.x|20260822
H|20260822|issue|id|name
I|NOTANUMBER|Broken
E|issue
'''

dirty = WORK / "drops" / "dirty.zip"
with zipfile.ZipFile(dirty, "w") as z:
    z.writestr("good.txt", SHAPE_B.format(n=0))
    for n in range(3):
        z.writestr(f"bad_{n}.txt", BROKEN)

strict = FeedSpec.from_yaml(ROOT / "feeds" / "pipe-tagged-feed.yaml")
strict.source.pattern = str(dirty)
strict.target.table = "sandbox.strict"
strict.policy.max_reject_ratio = 0.01        # allow 1% bad

gated = run(strict, WORK / "ws", workers=2)

print(f"status         : {gated.status}")
print(f"good rows      : {gated.rows}")
print(f"bad rows       : {gated.rejects}")
print(f"bad proportion : {gated.reject_ratio:.0%}   (limit was {strict.policy.max_reject_ratio:.0%})")
print(f"saved          : {gated.commit or 'NOTHING -- refused'}")
print(f"reason         : {gated.errors[0]['message']}")

50% of the rows were bad, so **nothing was saved**. Not the good rows either.

That's the point: quietly loading half a broken file is worse than failing, because
someone finds out weeks later. Raise the limit only if you genuinely mean it.

Now the same drop with a limit that allows it, so we can look at the bad rows:

In [ ]:
lenient = FeedSpec.from_yaml(ROOT / "feeds" / "pipe-tagged-feed.yaml")
lenient.source.pattern = str(dirty)
lenient.target.table = "sandbox.lenient"
lenient.policy.max_reject_ratio = 0.9        # allow it this time

kept = run(lenient, WORK / "ws", workers=2)
print(f"good rows saved: {kept.rows}   bad rows quarantined: {kept.rejects}\n")

bad = scan(WORK / "ws" / "warehouse", "sandbox.lenient_rejects")
print(bad.select(["id", "name", "_src_file", "_src_line_no", "_reject_reason"]))

The bad rows kept their **original text** — `id` is still `'NOTANUMBER'`, not a
null. So you can see exactly what arrived, on exactly which line.

**This is the feedback loop.** When a file that passed on a 100-row sample fails in
production, you query this table, look at the real failing rows, and fix the
description or the parser. Add the failing shape to a test fixture so it can't
come back.

---
## 10. What's in the warehouse

Every run is recorded, so you can answer "did yesterday's drop land?" without
digging.

In [ ]:
from ffe.io.sink import catalog

cat = catalog(WORK / "ws" / "warehouse")
print(f"{'table':26} {'rows':>6}  {'files':>5}  snapshots")
for (ns,) in cat.list_namespaces():
    for _, name in cat.list_tables(ns):
        t = cat.load_table(f"{ns}.{name}")
        s = t.current_snapshot()
        rows = int(s.summary.get("total-records", 0)) if s else 0
        files = int(s.summary.get("total-data-files", 0)) if s else 0
        print(f"{ns + '.' + name:26} {rows:>6}  {files:>5}  {len(t.metadata.snapshots)}")

In [ ]:
from ffe.io.ledger import Ledger

ledger = Ledger(WORK / "ws" / "ledger.db")
history = ledger.conn.execute(
    "SELECT feed, table_name, status, members, rows, rejects FROM jobs ORDER BY started"
).fetchall()

print(f"{'feed':20} {'table':18} {'status':8} {'files':>5} {'rows':>6} {'bad':>5}")
for feed_name, table, status, members, rows, rejects in history:
    print(f"{feed_name:20} {table:18} {status:8} {members:>5} {rows or 0:>6} {rejects or 0:>5}")

---
## 11. A normal CSV needs none of this

For a file that's just a file, there's no structure to describe — say `native` and
name the types you want.

In [ ]:
from ffe.core.spec import Coerce, NativeParser

plain = WORK / "plain.csv"
plain.write_text("id,name,score\n1,Abhishek,10\n2,Nilanjana,30\nX,Broken,5\n")

res = parse(
    NativeParser(coerce=Coerce(schema_override={"id": "int64", "score": "int64"})),
    plain.read_bytes(),
)
print("good rows:"); print(res.frame)
print("bad rows:");  print(res.rejects.select(["id", "name", "_src_line_no", "_reject_reason"]))

Note that types are only what you asked for. Nothing is guessed — on purpose, so a
column can't silently change type next month when a new file arrives.

---
## 12. Clean up

Deletes only `_notebook_demo/`. Your feeds, plugins and real warehouse are untouched.

In [ ]:
shutil.rmtree(WORK, ignore_errors=True)
print("cleaned up", WORK)

---
## Where to go next

- **[docs/GETTING-STARTED.md](../docs/GETTING-STARTED.md)** — the same ground, step by step, for your own files
- **`ffe profile <your file>`** — start here with a real sample
- **`plugins/acme_positions.py`** — a worked example for files too odd to describe in YAML
- **[docs/DESIGN.md](../docs/DESIGN.md)** — why it's built this way, and the measurements behind the choices